In [ ]:
from models.factory import ModelFactory

model_wrapper = ModelFactory.create(
    model_type="multimode_coconut",
    model_path=r"checkpoints/llama32-1b_gsm_multimode/checkpoint_10",
    device="cuda",
    model_id=r"D:/Researches/InterpreteJLens/base_models/Llama-3.2-1B-Instruct",
)

In [ ]:
print(type(model_wrapper))
print(type(model_wrapper.model))
print(type(model_wrapper.model.base_causallm))

In [ ]:
from pathlib import Path
import pandas as pd
import random
import json

# ============================================================
# Read local WikiText-103 parquet files
# and save 1000 calibration prompts
# ============================================================

DATA_DIR = Path(
    r"D:\Researches\InterpreteJLens\data\wikitext103"
)

PARQUET_FILES = [
    DATA_DIR / "train-00000-of-00002.parquet",
    DATA_DIR / "train-00001-of-00002.parquet",
]

OUT = Path(
    r"D:\Researches\InterpreteJLens\jlens_outputs"
)
OUT.mkdir(parents=True, exist_ok=True)

SAVE_PATH = OUT / "wikitext103_calibration_1000_1.json"


tokenizer = model_wrapper.tokenizer


# ------------------------------------------------------------
# Load and concatenate parquet files
# ------------------------------------------------------------

dfs = [
    pd.read_parquet(path)
    for path in PARQUET_FILES
]

df = pd.concat(
    dfs,
    ignore_index=True
)

print("rows:", len(df))
print("columns:", df.columns.tolist())

print("\nFirst rows:")
print(df.head())

In [ ]:
# ============================================================
# Build exactly 1000 WikiText calibration prompts
# ============================================================

import random

N_PROMPTS = 1000
MIN_TOKENS = 64
MAX_TOKENS = 128
RANDOM_SEED = 43 ## 42-51
SAVE_PATH = OUT / "wikitext103_calibration_1000_1.json"

rng = random.Random(RANDOM_SEED)

# ------------------------------------------------------------
# First shuffle WikiText rows
# ------------------------------------------------------------

texts = [
    text.strip()
    for text in df["text"]
    if isinstance(text, str)
    and text.strip()
    and not (
        text.strip().startswith("=")
        and text.strip().endswith("=")
    )
]

rng.shuffle(texts)

selected = []


# ------------------------------------------------------------
# Process rows until we get 1000 valid prompts
# ------------------------------------------------------------

for text in texts:

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    # Too short
    if len(token_ids) < MIN_TOKENS:
        continue

    # If longer than MAX_TOKENS,
    # randomly select one contiguous passage
    if len(token_ids) > MAX_TOKENS:

        max_start = len(token_ids) - MAX_TOKENS

        start = rng.randint(
            0,
            max_start
        )

        chunk_ids = token_ids[
            start:start + MAX_TOKENS
        ]

    else:
        chunk_ids = token_ids

    chunk_text = tokenizer.decode(
        chunk_ids,
        skip_special_tokens=True
    ).strip()

    if not chunk_text:
        continue

    selected.append({
        "text": chunk_text,
        "n_tokens": len(chunk_ids)
    })

    # Stop immediately once we have 1000
    if len(selected) >= N_PROMPTS:
        break


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

if len(selected) < N_PROMPTS:
    raise RuntimeError(
        f"Only found {len(selected)} valid prompts."
    )

calibration_prompts = [
    x["text"]
    for x in selected
]

lengths = [
    x["n_tokens"]
    for x in selected
]

print("num prompts:", len(selected))
print("min tokens:", min(lengths))
print("mean tokens:", sum(lengths) / len(lengths))
print("max tokens:", max(lengths))

In [ ]:
# ============================================================
# Save the 1000 calibration prompts
# ============================================================

import json

with open(
    SAVE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "dataset": "WikiText-103",
            "split": "train",
            "seed": RANDOM_SEED,
            "n_prompts": N_PROMPTS,
            "min_tokens": MIN_TOKENS,
            "max_tokens": MAX_TOKENS,
            "prompts": selected
        },
        f,
        ensure_ascii=False,
        indent=2
    )


print("saved:", SAVE_PATH)
print("num prompts:", len(calibration_prompts))

for i in range(5):

    print(f"\nPROMPT {i}")
    print(calibration_prompts[i])
    print(
        "tokens:",
        selected[i]["n_tokens"]
    )

In [ ]:
import json
from pathlib import Path
import jlens

OUT = Path(
    r"D:\Researches\InterpreteJLens\jlens_outputs"
)

PROMPT_PATH = OUT / "wikitext103_calibration_1000_1.json"

CHECKPOINT_PATH = (
    OUT / "coconut_jlens_wikitext103_1000_1_checkpoint.pt"
)

LENS_PATH = (
    OUT / "coconut_jlens_wikitext103_1000_1.pt"
)


with open(
    PROMPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    calibration_data = json.load(f)

calibration_prompts = [
    x["text"]
    for x in calibration_data["prompts"]
]

print("Loaded prompts:", len(calibration_prompts))


coconut_hf_model = model_wrapper.model.base_causallm
tokenizer = model_wrapper.tokenizer

coconut_lens_model = jlens.from_hf(
    coconut_hf_model,
    tokenizer,
    compile=False,
)

print("n_layers:", coconut_lens_model.n_layers)
print("d_model:", coconut_lens_model.d_model)


coconut_lens = jlens.fit(
    coconut_lens_model,
    prompts=calibration_prompts,
    max_seq_len=128,
    skip_first=4,
    checkpoint_path=str(CHECKPOINT_PATH),
    resume=False,
)

print("J-lens fit finished.")

coconut_lens.save(
    str(LENS_PATH)
)

print("Saved calibrated J-lens to:")
print(LENS_PATH)

print("Source layers:")
print(coconut_lens.source_layers)